In [1]:
from pydub import AudioSegment
import os

input_path = "/Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/10.2.2019_radio15_TR_SAA_SA_VR.wav"
output_dir = "/Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented"
segment_duration = 20 * 1000  # 20 seconds in milliseconds

os.makedirs(output_dir, exist_ok=True)
audio = AudioSegment.from_wav(input_path)

for i, start in enumerate(range(0, len(audio), segment_duration)):
    end = min(start + segment_duration, len(audio))
    segment = audio[start:end]
    segment.export(os.path.join(output_dir, f"segment_{i:03}.wav"), format="wav")


In [2]:
import tqdm as notebook_tqdm
import torch
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2ForCTC, Wav2Vec2Processor

model = Wav2Vec2ForCTC.from_pretrained("enenlhet-asr/enenlhet-wav2vec2-model")

# Load processor
processor = Wav2Vec2Processor.from_pretrained("enenlhet-asr/enenlhet-wav2vec2-model", return_attention_mask=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()


/Users/sjhuskey/miniconda3/envs/whisper/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [3]:
import torchaudio
import torch

def transcribe_wav2vec2(audio_path):

    # Load and preprocess
    speech_array, sampling_rate = torchaudio.load(audio_path)
    if speech_array.shape[0] > 1:
        speech_array = torch.mean(speech_array, dim=0, keepdim=True)
    if sampling_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
        speech_array = resampler(speech_array)
        sampling_rate = 16000
    speech_array = speech_array.squeeze(0)

    # Tokenize
    inputs = processor(speech_array.numpy(), sampling_rate=16000, return_tensors="pt")
    input_values = inputs.input_values.to(model.device)
    attention_mask = inputs.attention_mask.to(model.device)

    # Inference with attention mask
    with torch.no_grad():
        logits = model(input_values, attention_mask=attention_mask).logits

    # Decode
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]

    return transcription


In [5]:
import glob
import os

segment_files = sorted(glob.glob("/Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented/*.wav"))
results = []

for fpath in segment_files[0:10]:
    print(f"Transcribing: {fpath}")
    transcript = transcribe_wav2vec2(fpath)
    results.append((os.path.basename(fpath), transcript))

for filename, transcript in results:
    print(f"{filename}: {transcript}")


Transcribing: /Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented/segment_000.wav
Transcribing: /Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented/segment_001.wav
Transcribing: /Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented/segment_002.wav
Transcribing: /Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented/segment_003.wav
Transcribing: /Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented/segment_004.wav
Transcribing: /Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented/segment_005.wav
Transcribing: /Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented/segment_006.wav
Transcribing: /Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented/segment_007.wav
Transcribing: /Users/sjhuskey/Downloads/No_transcripts_yet/radio-station-recordings/segmented/segment_008.wav
Transcribi

In [ ]:
import csv

with open("enenlhet_wav2vec2_transcriptions.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Segment", "Transcription"])
    writer.writerows(results)